# Inspect Run16 + Run12-Run15 U64 Sweep

This notebook compares the new centered U64 runs. `run16` is the clean baseline rerun, and `run12`-`run15` are one-change-at-a-time tests against it.

| Run | Config | Change | Epochs | Main question |
|---|---|---|---:|---|
| `run16` | `config_run16_u64_baseline.yaml` | fresh baseline: mean-centered maxabs, roll + flip, `zthin=4` | 120 | clean reference baseline |
| `run12` | `config_run12_u64_median_center.yaml` | median center instead of mean center | 120 | does robust centering help? |
| `run13` | `config_run13_u64_low_lr.yaml` | lower LR, `5e-5` | 120 | does smoother optimization help? |
| `run14_short` | `config_run14_short.yaml` | D4 augmentation, short-name folder | 120 | D4 result if this is the submitted job |
| `run14_u64` | `config_run14_u64_d4_aug.yaml` | D4 augmentation, original sweep-name folder | 120 | D4 result if this is the submitted job |
| `run15` | `config_run15_zthin2.yaml` | more slices, `zthin=2` | 120 | does more training data help? |

All runs keep the architecture fixed: U64, `layers_per_block=2`, DDPM, `log=True`, and `normalization=centered_maxabs`.

Important: `run14_short` and `run14_u64` are the same experimental idea. Keep both in the notebook only because you may have submitted either one. When interpreting the sweep, do not count them as two independent ideas.


## What You Actually Run

You do **not** run the YAML file directly. The YAML is the configuration. On Great Lakes, submit the matching Slurm batch script.

Recommended sweep commands:

```bash
cd /home/jiamingp/Diffusion_model
sbatch batch_scripts/train_diffusion_run16_u64_baseline.sbatch
sbatch batch_scripts/train_diffusion_run12_u64_median_center.sbatch
sbatch batch_scripts/train_diffusion_run13_u64_low_lr.sbatch
sbatch batch_scripts/train_diffusion_run14_u64_d4_aug.sbatch
sbatch batch_scripts/train_diffusion_run15_zthin2.sbatch
```

Optional duplicate D4 script, only if you intentionally used the short-name version:

```bash
sbatch batch_scripts/train_diffusion_run14_short.sbatch
```

`run16` replaces `run8` as the baseline for this notebook because `run8` already had a final checkpoint and should not be resubmitted.


In [ ]:
from pathlib import Path
import sys
import copy
import json

import numpy as np
import torch
import yaml
import matplotlib.pyplot as plt

PROJECT_DIR = Path("/home/jiamingp/Diffusion_model")
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd()

COSMO_DIFFUSION_DIR = PROJECT_DIR / "cosmo_diffusion"
if str(COSMO_DIFFUSION_DIR) not in sys.path:
    sys.path.insert(0, str(COSMO_DIFFUSION_DIR))

from cosmodiff import utils
from cosmodiff.optim import generate

print("cosmodiff utils:", utils.__file__)
if "centered_maxabs" not in open(utils.__file__).read():
    raise RuntimeError("This kernel is using an old utils.py. Sync the patched file to Great Lakes, then restart the kernel.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PROJECT_DIR:", PROJECT_DIR)
print("DEVICE:", DEVICE)
if DEVICE.type != "cuda":
    print("WARNING: use a GPU kernel on Great Lakes before loading CUDA-saved checkpoints.")


In [ ]:
RUNS = {
    "run16": {
        "config": PROJECT_DIR / "configs" / "great_lakes" / "config_run16_u64_baseline.yaml",
        "label": "run16: fresh baseline mean center",
        "color": "tab:green",
        "style": "-",
    },
    "run12": {
        "config": PROJECT_DIR / "configs" / "great_lakes" / "config_run12_u64_median_center.yaml",
        "label": "run12: median center",
        "color": "tab:blue",
        "style": "-",
    },
    "run13": {
        "config": PROJECT_DIR / "configs" / "great_lakes" / "config_run13_u64_low_lr.yaml",
        "label": "run13: lower LR",
        "color": "tab:red",
        "style": "-",
    },
    "run14_short": {
        "config": PROJECT_DIR / "configs" / "great_lakes" / "config_run14_short.yaml",
        "label": "run14_short: D4 aug",
        "color": "tab:purple",
        "style": "--",
    },
    "run14_u64": {
        "config": PROJECT_DIR / "configs" / "great_lakes" / "config_run14_u64_d4_aug.yaml",
        "label": "run14_u64: D4 aug",
        "color": "tab:pink",
        "style": "-",
    },
    "run15": {
        "config": PROJECT_DIR / "configs" / "great_lakes" / "config_run15_zthin2.yaml",
        "label": "run15: zthin=2",
        "color": "tab:orange",
        "style": "-",
    },
}

N_GEN = 20
N_REAL = 128
NBINS = 25
SEED = 123

for run_name, meta in RUNS.items():
    print(run_name, "config exists:", meta["config"].exists(), meta["config"])


## Check The Configs

This table is the sanity check before comparing outputs. For a clean experiment, most columns should match across runs, and only the intended variable should change.


In [ ]:
def load_config(path):
    with open(path) as f:
        return yaml.safe_load(f)

rows = []
for run_name, meta in RUNS.items():
    cfg = load_config(meta["config"])
    data = cfg["data"]
    model = cfg["model"]["kwargs"]
    opt = cfg["optimizer"]["kwargs"]
    train = cfg["train"]
    rows.append({
        "run": run_name,
        "epochs": train["num_epochs"],
        "checkpoint_every": train.get("checkpoint_every_n_epochs"),
        "zthin": data.get("zthin"),
        "n_samples": data.get("n_samples"),
        "normalization": data.get("normalization"),
        "center": (data.get("norm_kwargs") or {}).get("center"),
        "lr": opt.get("lr"),
        "batch_size": train.get("batch_size"),
        "grad_accum": train.get("gradient_accumulation_steps"),
        "channels": model.get("block_out_channels"),
        "layers_per_block": model.get("layers_per_block"),
        "augmentations": list((cfg.get("augmentations") or {}).keys()),
        "output_dir": cfg["io"]["output_dir"],
    })

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except Exception:
    for row in rows:
        print(json.dumps(row, indent=2))


## Raw Data Shape And `zthin` Audit

This checks what the configured `.npy` file actually contains before normalization and reshaping.

Key distinction:

- `z=0.0` in the filename means redshift zero.
- `zthin` means thinning along the spatial depth axis inside each 3D box.
- `n_samples: null` means use all simulations in the file.

For a raw array shaped like `(N_sim, N_z, N_x, N_y)`, the 2D training count is approximately:

```text
N_train_2D = N_sim_used * ceil(N_z / zthin)
```


In [ ]:
def audit_raw_data_shape(config_path):
    cfg = load_config(config_path)
    data_cfg = cfg["data"]
    img_path = Path(data_cfg["img_path"])
    zthin = int(data_cfg.get("zthin", 1))
    n_samples = data_cfg.get("n_samples", None)
    two_dim = bool(data_cfg.get("two_dim", True))

    arr = np.load(img_path, mmap_mode="r")
    raw_shape = tuple(arr.shape)
    n_sim_total = raw_shape[0]
    n_sim_used = n_sim_total if n_samples is None else min(int(n_samples), n_sim_total)

    if len(raw_shape) >= 4:
        n_z = raw_shape[1]
        kept_z_indices = list(range(0, n_z, zthin))
        n_z_kept = len(kept_z_indices)
    else:
        n_z = None
        kept_z_indices = []
        n_z_kept = None

    if two_dim and n_z_kept is not None:
        n_training_images = n_sim_used * n_z_kept
    else:
        n_training_images = n_sim_used

    return {
        "config": Path(config_path).name,
        "img_file": img_path.name,
        "raw_shape": raw_shape,
        "n_sim_total_in_file": n_sim_total,
        "n_samples_config": n_samples,
        "n_sim_used": n_sim_used,
        "redshift_from_filename": "z=0.0" if "z=0.0" in img_path.name else "check filename",
        "spatial_z_slices_raw": n_z,
        "zthin": zthin,
        "spatial_z_slices_kept": n_z_kept,
        "first_kept_z_indices": kept_z_indices[:10],
        "last_kept_z_index": None if not kept_z_indices else kept_z_indices[-1],
        "two_dim": two_dim,
        "expected_2d_training_images": n_training_images,
    }

shape_rows = []
for run_name, meta in RUNS.items():
    row = audit_raw_data_shape(meta["config"])
    row["run"] = run_name
    shape_rows.append(row)

try:
    import pandas as pd
    display(pd.DataFrame(shape_rows))
except Exception:
    for row in shape_rows:
        print(json.dumps(row, indent=2))


## Check What Finished

This cell reports whether each run has checkpoints yet and whether the latest checkpoint reaches the target final epoch. Missing runs are skipped later, so you can use the notebook while jobs are still running.


In [ ]:
def latest_checkpoint_from_config(cfg):
    output_dir = Path(cfg["io"]["output_dir"])
    ckpt = utils.find_latest_checkpoint(str(output_dir))
    return output_dir, ckpt

status_rows = []
for run_name, meta in RUNS.items():
    cfg = load_config(meta["config"])
    output_dir, ckpt = latest_checkpoint_from_config(cfg)
    target_last_epoch = cfg["train"]["num_epochs"] - 1
    if ckpt is None:
        loaded_epoch = None
        complete = False
    else:
        loaded_epoch = int(Path(ckpt).name.split("-")[-1])
        complete = loaded_epoch >= target_last_epoch
    status_rows.append({
        "run": run_name,
        "output_dir": str(output_dir),
        "latest_checkpoint": None if ckpt is None else Path(ckpt).name,
        "latest_epoch": loaded_epoch,
        "target_epoch": target_last_epoch,
        "complete": complete,
    })

try:
    import pandas as pd
    display(pd.DataFrame(status_rows))
except Exception:
    for row in status_rows:
        print(json.dumps(row, indent=2))


## Load Checkpoints

The target training length is `num_epochs: 120`, so the final checkpoint should usually be `checkpoint-epoch-0119`. If this loads an earlier checkpoint, treat the plots as undertrained diagnostics.


In [ ]:
def checkpoint_epoch(path):
    return int(Path(path).name.split("-")[-1])


def latest_metrics_path(output_dir, checkpoint_dir=None):
    candidates = []
    if checkpoint_dir is not None:
        ckpt_metrics = Path(checkpoint_dir) / "metrics.json"
        if ckpt_metrics.exists():
            candidates.append(ckpt_metrics)
    candidates.extend(Path(output_dir).glob("metrics_epoch_*.json"))
    if not candidates:
        return None
    return sorted(candidates, key=lambda p: p.stat().st_mtime)[-1]

run_state = {}
for run_name, meta in RUNS.items():
    cfg = load_config(meta["config"])
    output_dir = Path(cfg["io"]["output_dir"])
    ckpt = utils.find_latest_checkpoint(str(output_dir))

    print(f"\n{run_name}")
    print("  output_dir:", output_dir)
    print("  latest checkpoint:", ckpt)
    if ckpt is None:
        print("  SKIP: no checkpoint yet. Train this run first.")
        continue

    model, scheduler, optimizer, lr_scheduler, augmentations = utils.load_checkpoint(ckpt)
    model.to(DEVICE)
    model.eval()

    metrics_path = latest_metrics_path(output_dir, ckpt)
    metrics = utils.read_metrics(str(metrics_path)) if metrics_path is not None else None
    target_last_epoch = cfg["train"]["num_epochs"] - 1
    loaded_epoch = checkpoint_epoch(ckpt)

    print("  loaded epoch:", loaded_epoch)
    print("  target final epoch:", target_last_epoch)
    if loaded_epoch < target_last_epoch:
        print("  WARNING: checkpoint is not final yet; samples may be undertrained.")
    print("  metrics:", metrics_path)
    print("  augmentation restored:", augmentations)

    run_state[run_name] = {
        "config": cfg,
        "output_dir": output_dir,
        "checkpoint": Path(ckpt),
        "model": model,
        "scheduler": scheduler,
        "metrics": metrics,
        "metrics_path": metrics_path,
    }

print("\nloaded runs:", list(run_state))


## Training Curves

Use this first. If a run has unstable loss or a very different LR curve, that matters before interpreting images or `P(k)`.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for run_name, state in run_state.items():
    metrics = state["metrics"]
    if metrics is None:
        continue
    color = RUNS[run_name]["color"]
    label = RUNS[run_name]["label"]

    if "epoch_loss" in metrics:
        axes[0].plot(metrics["epoch_loss"], label=label, color=color, ls=RUNS[run_name].get("style", "-"), lw=2)
    if "epoch_lr" in metrics:
        axes[1].plot(metrics["epoch_lr"], label=label, color=color, ls=RUNS[run_name].get("style", "-"), lw=2)

axes[0].set_title("Epoch loss")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("noise-prediction MSE")
axes[0].legend()

axes[1].set_title("Learning rate")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("lr")
axes[1].legend()
fig.tight_layout()
plt.show()


## Load Real Reference Data

Each run's real data are loaded using that run's own YAML normalization. This is important because `run12` centers by median and `run15` uses more slices.


In [ ]:
real_images = {}
for run_name, state in run_state.items():
    data_config = copy.deepcopy(state["config"])
    data_config.setdefault("global", {})["device"] = "cpu"
    data_config.setdefault("data", {})["keep_on_cpu"] = True

    dataset = utils.parse_config_data(data_config)
    imgs = dataset.arrays.detach().cpu()

    if len(imgs) > N_REAL:
        idx = torch.linspace(0, len(imgs) - 1, N_REAL).long()
        imgs = imgs[idx]

    real_images[run_name] = imgs
    print(f"{run_name}: real_images shape = {tuple(imgs.shape)}")
    print(f"  range = [{imgs.min().item():.4f}, {imgs.max().item():.4f}]")
    print(f"  mean/std = {imgs.mean().item():.6f} / {imgs.std().item():.6f}")
    print(f"  frac(|x| >= 0.999) = {((imgs.abs() >= 0.999).float().mean().item()):.8e}")


## Generate Samples

This uses each checkpoint's saved DDPM scheduler. We are not changing the sampler in this notebook; this is testing training/data choices.


In [ ]:
generated = {}
for run_name, state in run_state.items():
    model = state["model"]
    scheduler = state["scheduler"]

    g = torch.Generator(device=DEVICE).manual_seed(SEED)
    with torch.no_grad():
        samples = generate(
            model,
            scheduler,
            batch_size=N_GEN,
            image_shape=(1, 128, 128),
            device=DEVICE,
            generator=g,
        ).detach().cpu()

    generated[run_name] = samples
    print(f"\n{run_name}")
    print(f"  generated shape = {tuple(samples.shape)}")
    print(f"  range = [{samples.min().item():.4f}, {samples.max().item():.4f}]")
    print(f"  mean/std = {samples.mean().item():.6f} / {samples.std().item():.6f}")
    print(f"  frac(|x| >= 0.999) = {((samples.abs() >= 0.999).float().mean().item()):.8e}")


## Visual Samples

The image display uses a shared percentile color scale for the real and generated samples in the same run. This avoids the “everything looks dim” issue caused by forcing `vmin=-1, vmax=1` when most pixels do not occupy the full range.


In [ ]:
def run_vlim(*tensors, q_low=0.5, q_high=99.5):
    vals = torch.cat([t.detach().cpu().flatten() for t in tensors])
    return torch.quantile(vals, torch.tensor([q_low / 100, q_high / 100])).tolist()


def plot_real_fake_grid(run_name, n=8):
    real = real_images[run_name][:n]
    fake = generated[run_name][:n]
    vmin, vmax = run_vlim(real, fake)

    fig, axes = plt.subplots(2, n, figsize=(1.8 * n, 3.8))
    for i in range(n):
        axes[0, i].imshow(real[i, 0], cmap="magma", vmin=vmin, vmax=vmax)
        axes[0, i].axis("off")
        axes[1, i].imshow(fake[i, 0], cmap="magma", vmin=vmin, vmax=vmax)
        axes[1, i].axis("off")

    axes[0, 0].set_ylabel("real", fontsize=12)
    axes[1, 0].set_ylabel("generated", fontsize=12)
    fig.suptitle(f"{RUNS[run_name]['label']}  |  display range [{vmin:.3f}, {vmax:.3f}]", y=1.02)
    fig.tight_layout()
    plt.show()

for run_name in run_state:
    plot_real_fake_grid(run_name, n=min(8, N_GEN, N_REAL))


## Pixel Histograms

This is the quickest check of whether the generated one-point distribution matches the real data after the same normalization.


In [ ]:
def summarize_tensor(name, x):
    x = x.detach().cpu().flatten()
    qs = torch.quantile(x, torch.tensor([0.0, 0.001, 0.01, 0.1, 0.5, 0.9, 0.99, 0.999, 1.0]))
    print(name)
    print(f"  min/median/max = {qs[0].item():.4f} / {qs[4].item():.4f} / {qs[-1].item():.4f}")
    print(f"  q99/q99.9      = {qs[6].item():.4f} / {qs[7].item():.4f}")
    print(f"  mean/std       = {x.mean().item():.6f} / {x.std().item():.6f}")
    print(f"  frac(|x|>=.999) = {(x.abs() >= 0.999).float().mean().item():.8e}")

active_runs = list(run_state)
fig, axes = plt.subplots(1, len(active_runs), figsize=(6 * len(active_runs), 4), sharey=True)
axes = np.atleast_1d(axes)
bins = np.linspace(-1, 1, 120)

for ax, run_name in zip(axes, active_runs):
    real = real_images[run_name].numpy().ravel()
    fake = generated[run_name].numpy().ravel()
    ax.hist(real, bins=bins, density=True, histtype="step", lw=2, color="black", label="real")
    ax.hist(fake, bins=bins, density=True, histtype="step", lw=2, color=RUNS[run_name]["color"], linestyle=RUNS[run_name].get("style", "-"), label="generated")
    ax.set_yscale("log")
    ax.set_title(RUNS[run_name]["label"])
    ax.set_xlabel("normalized pixel value")
    ax.legend()
axes[0].set_ylabel("density")
fig.tight_layout()
plt.show()

for run_name in active_runs:
    print("\n" + run_name)
    summarize_tensor("real", real_images[run_name])
    summarize_tensor("generated", generated[run_name])


## Radial 2D Power Spectrum

The main diagnostic is:

```text
generated P(k) / real P(k)
```

computed in the same transformed space for each run.


In [ ]:
def radial_power_spectrum_2d(field, nbins=25):
    field = np.asarray(field, dtype=np.float64)
    field = field - field.mean()

    fft = np.fft.fftn(field)
    power = (fft * fft.conj()).real / field.size

    ky = np.fft.fftfreq(field.shape[0]) * field.shape[0]
    kx = np.fft.fftfreq(field.shape[1]) * field.shape[1]
    kkx, kky = np.meshgrid(kx, ky)
    kvals = np.sqrt(kkx**2 + kky**2)

    valid = kvals > 0
    edges = np.linspace(kvals[valid].min(), kvals[valid].max(), nbins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])

    pk = np.full(nbins, np.nan)
    for i in range(nbins):
        mask = (kvals >= edges[i]) & (kvals < edges[i + 1])
        if mask.any():
            pk[i] = power[mask].mean()
    return pk, centers


def batch_power_spectra(images, nbins=25):
    arr = images.detach().cpu().numpy()
    pks = []
    kbins = None
    for img in arr:
        pk, kbins = radial_power_spectrum_2d(img[0], nbins=nbins)
        pks.append(pk)
    return np.asarray(pks), kbins

pk = {}
for run_name in run_state:
    real_pk, kbins = batch_power_spectra(real_images[run_name], NBINS)
    fake_pk, _ = batch_power_spectra(generated[run_name], NBINS)
    pk[run_name] = {"real": real_pk, "fake": fake_pk, "kbins": kbins}

print("computed P(k) for:", list(pk.keys()))


In [ ]:
def band_summary(ratio):
    finite = np.where(np.isfinite(ratio))[0]
    if len(finite) == 0:
        return np.nan, np.nan, np.nan
    thirds = np.array_split(finite, 3)
    return tuple(float(np.nanmean(ratio[t])) for t in thirds)

print("Mean P(k) ratio relative to each run's real-data mean:")
for run_name in pk:
    real_mean = np.nanmean(pk[run_name]["real"], axis=0)
    fake_mean = np.nanmean(pk[run_name]["fake"], axis=0)
    ratio = fake_mean / np.clip(real_mean, 1e-30, None)
    low, mid, high = band_summary(ratio)
    print(f"  {run_name:6s} | low-k={low:.3f}  mid-k={mid:.3f}  high-k={high:.3f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for run_name in pk:
    color = RUNS[run_name]["color"]
    kbins = pk[run_name]["kbins"]
    real_mean = np.nanmean(pk[run_name]["real"], axis=0)
    real_std = np.nanstd(pk[run_name]["real"], axis=0)
    fake_mean = np.nanmean(pk[run_name]["fake"], axis=0)

    axes[0].plot(kbins, real_mean, color=color, lw=2, ls=":", label=f"{run_name} real")
    axes[0].plot(kbins, fake_mean, color=color, lw=2, ls=RUNS[run_name].get("style", "-"), label=f"{run_name} generated")
    axes[0].fill_between(
        kbins,
        np.clip(real_mean - real_std, 1e-30, None),
        real_mean + real_std,
        color=color,
        alpha=0.12,
    )

axes[0].set_yscale("log")
axes[0].set_xlabel("k bin")
axes[0].set_ylabel("mean P(k)")
axes[0].set_title("Transformed-space P(k)")
axes[0].legend(fontsize=9)

for run_name in pk:
    color = RUNS[run_name]["color"]
    kbins = pk[run_name]["kbins"]
    real_mean = np.nanmean(pk[run_name]["real"], axis=0)
    fake_mean = np.nanmean(pk[run_name]["fake"], axis=0)
    ratio = fake_mean / np.clip(real_mean, 1e-30, None)
    axes[1].plot(kbins, ratio, color=color, lw=2.5, ls=RUNS[run_name].get("style", "-"), label=f"{run_name} generated / real")

axes[1].axhline(1.0, color="black", lw=1.5, ls=":")
axes[1].set_xlabel("k bin")
axes[1].set_ylabel("mean generated P(k) / real mean P(k)")
axes[1].set_title("Main diagnostic")
axes[1].legend()
fig.tight_layout()
plt.show()


## Quick Ranking

This is a rough scalar summary. Smaller is better. It combines the absolute log-error of the mean power-spectrum ratio and the mismatch in pixel mean/std. Use it only to guide attention; the plots are still the real diagnostic.


In [ ]:
scores = []
for run_name in pk:
    real_mean = np.nanmean(pk[run_name]["real"], axis=0)
    fake_mean = np.nanmean(pk[run_name]["fake"], axis=0)
    ratio = fake_mean / np.clip(real_mean, 1e-30, None)
    pk_log_mae = float(np.nanmean(np.abs(np.log10(np.clip(ratio, 1e-12, None)))))

    real_flat = real_images[run_name].flatten().float()
    fake_flat = generated[run_name].flatten().float()
    mean_err = float(abs(fake_flat.mean() - real_flat.mean()))
    std_err = float(abs(fake_flat.std() - real_flat.std()))

    scores.append({
        "run": run_name,
        "pk_log10_mae": pk_log_mae,
        "mean_abs_err": mean_err,
        "std_abs_err": std_err,
        "combined": pk_log_mae + mean_err + std_err,
    })

scores = sorted(scores, key=lambda r: r["combined"])
try:
    import pandas as pd
    display(pd.DataFrame(scores))
except Exception:
    for row in scores:
        print(json.dumps(row, indent=2))


## How To Interpret

Compare every run back to `run16`.

- If `run12` improves histogram and `P(k)`, median centering helps the skewed field.
- If `run13` improves results, U64 is sensitive to LR and `1e-4` may be too high.
- If either `run14_short` or `run14_u64` improves results, stronger D4 symmetry augmentation helps.
- If both D4 folders exist, use the one that actually reached the later checkpoint; they are not independent physics conclusions.
- If `run15` improves results, more slices from `zthin=2` help more than simply changing optimization.
- If D4 and `zthin=2` both help separately, the next useful run is probably `D4 + zthin=2` together.
